# fANOVA: Replications vs. Reference Run

This notebook:
1. Loads CP metrics for each of the replications individually
2. Runs a functional ANOVA (fANOVA) on each run separately
3. Compares each run's importance values with the reference (run 6) via Pearson and Spearman correlation

In [ ]:
%load_ext autoreload
%autoreload 1
%aimport fairness_multiverse

## 1. Configuration

In [ ]:
from pathlib import Path
import pandas as pd

REFERENCE_RUN = 6
RUNS = [17, 18, 19, 20, 21]
RUN_TO_SEED = {
    6: "reference",
    17: 80539,
    18: 1102,
    19: 47906,
    20: 378,
    21: 68131,
}

OUTPUT_DIR = Path("output")
main_cp_metric = "avg_size"  # "avg_size" or "cov_nongerman_female"

cols_design_dec = [
    "universe_training_year",
    "universe_training_size",
    "universe_scale",
    "universe_model",
    "universe_exclude_features",
    "universe_exclude_subgroups",
]

def run_display_label(run_no):
    seed = RUN_TO_SEED[run_no]
    return "Reference" if seed == "reference" else f"Seed {seed}"

## 2. Load CP Metrics per Run

In [ ]:
run_data = {}
for run_no in RUNS:
    cp_dir = OUTPUT_DIR / "runs" / str(run_no) / "cp_metrics"
    csv_files = list(cp_dir.glob("*.csv"))
    df_run = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
    run_data[run_no] = df_run
    print(f"{run_display_label(run_no)}: {df_run.shape[0]} universes")


## 3. Run fANOVA for Each Run Individually

In [ ]:
from fairness_multiverse.analysis import MultiverseFanova

fanova_results = {}
for run_no, df_run in run_data.items():
    df_run['universe_training_year'] = df_run['universe_training_year'].astype(str)

    print(f"Running fANOVA for {run_display_label(run_no)} ...")
    mf = MultiverseFanova(
        features=df_run[cols_design_dec],
        outcome=df_run[main_cp_metric],
    )
    df_imp = mf.quantify_importance()
    fanova_results[run_no] = df_imp
    print(f"  → {len(df_imp)} effects\n")


## 4. Load fANOVA Results from Run 6 (Reference)

In [ ]:
ref_path = OUTPUT_DIR / "runs" / str(REFERENCE_RUN) / f"fanova_importance_interactions-overall_{main_cp_metric}.csv"
df_importance_ref = pd.read_csv(ref_path, index_col=0)
print(f"{run_display_label(REFERENCE_RUN)}: {df_importance_ref.shape[0]} effects")
df_importance_ref.head(10)

## 5. Compare Each Run's Importance with Run 6

For each of runs 12–16 we join its fANOVA results with run 6 on effect key and compute Pearson and Spearman rank correlations.

In [ ]:
from scipy.stats import pearsonr, spearmanr

def make_effect_key(row):
    """Create a hashable key from the level_* columns."""
    levels = []
    for i in range(6):
        col = f"level_{i}"
        if col in row.index and pd.notna(row[col]) and row[col] != "":
            levels.append(str(row[col]))
    return " × ".join(sorted(levels))

# Add effect key to reference
df_importance_ref["effect_key"] = df_importance_ref.apply(make_effect_key, axis=1)

# Compute correlations per run
correlation_rows = []
merged_per_run = {}

for run_no, df_imp in fanova_results.items():
    df_imp = df_imp.copy()
    df_imp["effect_key"] = df_imp.apply(make_effect_key, axis=1)

    df_m = pd.merge(
        df_imp[["effect_key", "individual importance"]],
        df_importance_ref[["effect_key", "individual importance"]],
        on="effect_key",
        suffixes=(f"_run{run_no}", "_ref"),
    )
    merged_per_run[run_no] = df_m

    r_p, p_p = pearsonr(df_m[f"individual importance_run{run_no}"], df_m["individual importance_ref"])
    r_s, p_s = spearmanr(df_m[f"individual importance_run{run_no}"], df_m["individual importance_ref"])
    correlation_rows.append({
        "Seed": RUN_TO_SEED[run_no],
        "Pearson r": round(r_p, 4),
        "Spearman ρ": round(r_s, 4),
    })

tex_table_dir = OUTPUT_DIR / "analyses" / "var_imp_replications" / "tables"
tex_table_dir.mkdir(parents=True, exist_ok=True)

df_corr = pd.DataFrame(correlation_rows)
df_corr.to_latex(tex_table_dir / "df_corr.tex", float_format="%.4f", index=False)
df_corr

### Scatter Plots: Per-Seed Importance vs. Reference

In [ ]:
import matplotlib.pyplot as plt

scatter_pdf_dir = OUTPUT_DIR / "analyses" / "var_imp_replications" / "scatter_subfigs"
scatter_pdf_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, len(RUNS), figsize=(5 * len(RUNS), 5), sharey=True)

for ax, run_no in zip(axes, RUNS):
    df_m = merged_per_run[run_no]
    col_run = f"individual importance_run{run_no}"

    ax.scatter(
        df_m["individual importance_ref"],
        df_m[col_run],
        alpha=0.6,
        edgecolors="k",
        linewidths=0.5,
    )

    lims = [
        0,
        max(df_m["individual importance_ref"].max(), df_m[col_run].max()) * 1.05,
    ]
    ax.plot(lims, lims, "--", color="grey", linewidth=0.8)

    row = df_corr[df_corr["Seed"] == RUN_TO_SEED[run_no]].iloc[0]
    corr_title = f"{run_display_label(run_no)}\nr={row['Pearson r']:.3f}, ρ={row['Spearman ρ']:.3f}"
    ax.set_title(corr_title, fontsize=11)
    ax.set_xlabel("Importance — Reference", fontsize=10)
    ax.set_xlim(lims)
    ax.set_ylim(lims)

    # Save each subplot as a standalone PDF.
    fig_single, ax_single = plt.subplots(figsize=(5, 5))
    ax_single.scatter(
        df_m["individual importance_ref"],
        df_m[col_run],
        alpha=0.6,
        edgecolors="k",
        linewidths=0.5,
    )
    ax_single.plot(lims, lims, "--", color="grey", linewidth=0.8)
    ax_single.set_title(corr_title, fontsize=11)
    ax_single.set_xlabel("Importance — Reference", fontsize=10)
    ax_single.set_ylabel("Importance — Replication", fontsize=10)
    ax_single.set_xlim(lims)
    ax_single.set_ylim(lims)
    fig_single.tight_layout()

    seed = RUN_TO_SEED[run_no]
    fig_single.savefig(scatter_pdf_dir / f"importance_scatter_seed_{seed}.pdf")
    plt.close(fig_single)

axes[0].set_ylabel("Importance — Replication", fontsize=10)
plt.tight_layout()
plt.show()

print(f"Saved subplot PDFs to: {scatter_pdf_dir}")

### Top-10 Effects: Rank Comparison per Seed

In [ ]:
# Build a table of top-10 effects (by run 6 rank) with each run's rank
ref_ranked = (
    df_importance_ref
    .sort_values("individual importance", ascending=False)
    .reset_index(drop=True)
)
ref_ranked["rank_ref"] = range(1, len(ref_ranked) + 1)

top10_ref = ref_ranked.head(10)[["effect_key", "individual importance", "rank_ref"]].copy()
top10_ref.columns = ["Effect", "Importance (Reference)", "Rank (Reference)"]

for run_no in RUNS:
    df_imp = fanova_results[run_no].copy()
    df_imp["effect_key"] = df_imp.apply(make_effect_key, axis=1)
    run_ranked = df_imp.sort_values("individual importance", ascending=False).reset_index(drop=True)
    run_ranked["rank"] = range(1, len(run_ranked) + 1)
    rank_map = dict(zip(run_ranked["effect_key"], run_ranked["rank"]))
    top10_ref[f"Rank ({run_display_label(run_no)})"] = top10_ref["Effect"].map(rank_map)

top10_ref